# BioJEPA v0.7 Training Pipeline

In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from dataloader_v0_7 import EncoderLoader, ComposerLoader, TrainingLoader
from training_v0_7 import create_model, load_feature_banks, run_encoder_training, run_composer_training, run_ac_training, train_linear_decoder, maybe_compile
from config_v0_7 import EncoderTrainingConfig, ComposerTrainingConfig, ACTrainingConfig, DecoderConfig, DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report

## Device & Paths

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'short_checkpoint',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'short_eval_results'
)

using cuda


## Hyperparameters

In [ ]:
# Model architecture -- Trial 39 winning recipe (75/25 T22/T23 interpolation), composite 0.8853 at ep4
# Predictor dims decoupled from encoder: predictor HPO v1 winner (128/4/4, ~32.6% of encoder).
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,            # sim_coeff * std_to_sim_ratio (0.5069)
    cov_coeff=0.5158,           # sim_coeff * cov_to_sim_ratio (0.01028)
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)

# Encoder short: 10 epochs. Rampup matches HPO absolute timing (HPO used 50-ep schedule with
# warmup_pct=0.05 -> 2.5 ep absolute warmup). Setting warmup_pct=0.25 here reproduces the same
# 2.5 ep absolute warmup over the 10-ep horizon. context_coeff=0.0 matches HPO (ramp never fired
# in 4-ep trials). phase2_start_pct=0.8 adds production-style decay over the last 2 epochs.
encoder_cfg = EncoderTrainingConfig(
    epochs=10, lr=6.126e-5, batch_size=64,
    warmup_pct=0.25, weight_decay=0.05, phase2_start_pct=0.8,
    context_coeff=0.0, ema_final_momentum=None,
)

# Composer: v11 alignment HPO winner (trial #42, composite 0.6949 at 10k-epoch horizon)
composer_cfg = ComposerTrainingConfig(
    epochs=10000, lr=2.147e-4, batch_size=64, weight_decay=2.512e-3, temperature=7.559e-4, chemical_fraction=0.1,
)

# AC short: 5 epochs. Predictor HPO v1 winner (128/4/4, middle-of-plateau).
# lr 4e-3 tuned at batch 64; beta_nll_target 0.95 (plateau 0.9-1.0); mask anneal over last 2 ep.
ac_cfg = ACTrainingConfig(
    epochs=5, predictor_lr=4e-3, batch_size=64, beta_nll_target=0.95,
    mask_anneal_pct=0.4, mask_anneal_floor=0.0, composer_lr_mult=0.01,
)

# Decoder short: 5 epochs. Linear head converges in 3-5 epochs.
decoder_cfg = DecoderConfig(epochs=5, lr=1e-3, batch_size=32)

## Initialize Model & Data

In [4]:
model = create_model(model_cfg, device)
#model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 7,979,522
ACpredictor: 9,987,072
PerturbationComposer: 444,224


### Load pretraining model from checkpoint (for resuming)

In [ ]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_composer_final.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

# Predictor is trained fresh in the AC stage and its checkpoint shape (256/6) no longer matches the
# decoupled 128/4 predictor, so drop predictor.* before loading. strict=False tolerates the missing keys.
filtered = {k: v for k, v in checkpoint['model'].items() if not k.startswith('predictor.')}
keys = model.load_state_dict(filtered, strict=False)
assert all(k.startswith('predictor.') for k in keys.missing_keys) and not keys.unexpected_keys, keys
model = maybe_compile(model, USE_COMPILE)
keys

## Function to decompress and recompress a directory

In [ ]:
def decompress_npz(root_dir):
    root_dir = Path(root_dir)
    for src_path in root_dir.rglob('*.npz'):
        print(f'decompress {src_path}')
        with np.load(src_path, allow_pickle=False) as data:
            arrays = {k: data[k] for k in data.files}
        tmp_path = src_path.with_suffix('.tmp.npz')
        np.savez(tmp_path, **arrays)
        tmp_path.replace(src_path)

def compress_npz(root_dir):
    root_dir = Path(root_dir)
    for src_path in root_dir.rglob('*.npz'):
        print(f'compress {src_path}')
        with np.load(src_path, allow_pickle=False) as data:
            arrays = {k: data[k] for k in data.files}
        tmp_path = src_path.with_suffix('.tmp.npz')
        np.savez_compressed(tmp_path, **arrays)
        tmp_path.replace(src_path)

In [ ]:
# decompress_npz(data_cfg.data_root / 'encoder_t')

In [ ]:
# compress_npz(data_cfg.data_root / 'encoder_t')

## Stage 0: Composer Training

In [6]:
comp_train_loader = ComposerLoader(
    batch_size=composer_cfg.batch_size, 
    split='train', data_dir=data_cfg.data_root / 'pert_embd', 
    device=device,
    seed=1337, chemical_fraction=composer_cfg.chemical_fraction)
comp_val_loader = ComposerLoader(
    batch_size=composer_cfg.batch_size, 
    split='val', data_dir=data_cfg.data_root / 'pert_embd', 
    device=device)

found 1 shards for split train
  modality balancing: target chemical_fraction=10%
  adjusted total_samples=11829 (from 1 shards)
found 1 shards for split val


In [ ]:
align_results = run_composer_training(model, comp_train_loader, comp_val_loader, seq_banks, 
                                      target_bank, composer_cfg, device, data_cfg.checkpoint_dir, 
                                      use_amp=USE_AMP, use_fused_optimizer=USE_FUSED)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(align_results['loss_history'])
plt.yscale('log')
plt.title('Composer Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

### Composer Training Evals

In [7]:
eval_config = {
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': composer_cfg.batch_size, 'seed': SEED
}
model.eval()
align_eval_ctx = EvalContext(config=eval_config, data_root=data_cfg.data_root, checkpoint_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir)
align_eval_ctx._biojepa = model
align_eval_results = run_composer_evals(align_eval_ctx)
align_eval_ctx._biojepa = None

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

Using cuda
Loaded 10791 v0.7 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 128])
Encoded chemical sequences: torch.Size([188, 128])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 128])
seq_to_target_retrieval: dna_mrr=0.2503
cross_modality_target_consistency: Within=0.8058, Between=0.7631, Ratio=1.06x
seq_target_gap_analysis: dna_gap=0.83
paired_alignment_quality: dna_sim=0.8867
mode_sensitivity: Classification_acc=0.8024 (5.6x chance)
mode_semantic_consistency: semantic_gap=0.1307, cross_mode_mrr=0.0620
fusion_quality: Fused_var=0.5285, Seq_var=0.3931, Target_var=0.0891
missing_data_robustness: Fused_MRR=0.7077, Seq_only=0.2447, Target_only=0.9950
found 135 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|█████████████████████| 500/500 [00:03<00:00, 135.41it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0885, target_only=0.1508, fused=0.1697
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
action_vector_pathways DNA: ratio=1.00094357997238
Saved report to /home/ubuntu/data/v0_7/short_eval_results/composer_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.2503182431908235,
    'median_rank': 5.0,
    'mean_rank': 86.99463781749765,
    'n_queries': 10630,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.045343367826904984,
     '5': 0.5317027281279398,
     '10': 0.8947318908748824,
     '20': 0.9801505174035748,
     '50': 0.981749764816557}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 921,
   'n_within_pairs': 1408,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.805790364742279,
   'between_target_sim': 0.7630660832405091,
   'consistency_ratio': 1.055990277172762}},
 'seq_target_gap_analysis': {'target_variance': 10.862102508544922,
  'n_targets': 9975,
  'dna': {'seq_variance': 52.21984100341797,
   'centroid_distance': 3.0230116844177246,
   'mean_within_seq': 10.187459046577306,
   'mean_seq_to_target': 8.44333553314209,
   'gap_ratio': 0.8287970037021949,
   'n_sequences': 11643}},
 'paire

In [8]:
del comp_train_loader, comp_val_loader, align_eval_ctx
gc.collect()
torch.cuda.empty_cache()
model.train()

BioJepa(
  (student): OptimizedModule(
    (_orig_mod): CellStateEncoder(
      (expr_scaler): Linear(in_features=1, out_features=1, bias=False)
      (fourier_input_scaler): Linear(in_features=1, out_features=1, bias=False)
      (fourier_projection): GaussianFourierProjection()
      (film_generator): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=512, bias=True)
      )
      (total_count_proj): Linear(in_features=1, out_features=256, bias=True)
      (blocks): ModuleList(
        (0-5): 6 x CellStateBlock(
          (ln_1): RMSNorm()
          (attn): BioLinearAttention(
            (q_proj): Linear(in_features=256, out_features=256, bias=True)
            (k_proj): Linear(in_features=256, out_features=256, bias=True)
            (v_proj): Linear(in_features=256, out_features=256, bias=True)
            (c_proj): Linear(in_features=256, out_features=256, bias=True)

## Encoder Training

In [ ]:
enc_train_loader = EncoderLoader(
    batch_size=encoder_cfg.batch_size, 
    split='train', data_dir=data_cfg.data_root / 'encoder_t', 
    device=device, seed=1337)
enc_val_loader = EncoderLoader(
    batch_size=encoder_cfg.batch_size, 
    split='val', data_dir=data_cfg.data_root / 'encoder_t', 
    device=device, seed=1337)

In [ ]:
pt_results = run_encoder_training(model, enc_train_loader, enc_val_loader, encoder_cfg, device, data_cfg, model_cfg, use_amp=USE_AMP, use_fused_optimizer=USE_FUSED, eval_every_n_epochs=1)

Encoder training: 4065280 samples, 63520 steps/epoch, 635200 total steps
WSD schedule: warmup=158800 steps, phase2 starts at step 508160
Step 0 | val loss: 97.2201
Step 0 | Loss: 97.68965 | LR: 3.86e-10 | Phase: 1
Step 1000 | Loss: 71.88460 | LR: 3.86e-07 | Phase: 1
Step 2000 | Loss: 54.53749 | LR: 7.72e-07 | Phase: 1
Step 3000 | Loss: 56.42529 | LR: 1.16e-06 | Phase: 1
Step 4000 | Loss: 47.08051 | LR: 1.54e-06 | Phase: 1
Step 5000 | val loss: 46.4511
Step 5000 | Loss: 43.53844 | LR: 1.93e-06 | Phase: 1
Step 6000 | Loss: 43.50265 | LR: 2.31e-06 | Phase: 1
Step 7000 | Loss: 41.78257 | LR: 2.70e-06 | Phase: 1
Step 8000 | Loss: 40.18002 | LR: 3.09e-06 | Phase: 1
Step 9000 | Loss: 39.63225 | LR: 3.47e-06 | Phase: 1
Step 10000 | val loss: 39.0897
Step 10000 | Loss: 40.43414 | LR: 3.86e-06 | Phase: 1
Step 11000 | Loss: 38.89097 | LR: 4.24e-06 | Phase: 1
Step 12000 | Loss: 36.83238 | LR: 4.63e-06 | Phase: 1
Step 13000 | Loss: 39.25135 | LR: 5.02e-06 | Phase: 1
Step 14000 | Loss: 37.60069 | LR

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pt_results['loss_history'])
plt.yscale('log')
plt.title('Encoder Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

In [ ]:
if pt_results['epoch_evals']:
    evals_df = pd.DataFrame({e: d['metrics'] for e, d in pt_results['epoch_evals'].items()}).T
    evals_df.index = evals_df.index.astype(int)
    evals_df.index.name = 'epoch'
    evals_df.sort_index()

### Encoder Training Evals

In [ ]:
eval_config = {
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': encoder_cfg.batch_size, 'seed': SEED
}
model.eval()
eval_ctx = EvalContext(config=eval_config, data_root=data_cfg.data_root, checkpoint_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir)
eval_ctx._biojepa = model
pt_eval_results = run_encoder_evals(eval_ctx)
eval_ctx._biojepa = None

save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

In [ ]:
del enc_train_loader, enc_val_loader, eval_ctx
gc.collect()
torch.cuda.empty_cache()
model.train()

## Stage 3: AC Training

In [ ]:
train_loader = TrainingLoader(
    batch_size=ac_cfg.batch_size, 
    split='train', data_dir=data_cfg.data_root / 'predictor_t', 
    device=device)
val_loader = TrainingLoader(
    batch_size=ac_cfg.batch_size, 
    split='val', data_dir=data_cfg.data_root / 'predictor_t', 
    device=device)

full_results = run_ac_training(model, train_loader, val_loader, seq_banks, target_bank, ac_cfg, device, data_cfg.checkpoint_dir, use_amp=USE_AMP, use_fused_optimizer=USE_FUSED)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(full_results['loss_history'])
plt.title('AC Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

In [ ]:
del train_loader, val_loader
gc.collect()
torch.cuda.empty_cache()

## Linear Decoder Training

In [ ]:
decoder_train_loader = TrainingLoader(
    batch_size=decoder_cfg.batch_size, 
    split='train', data_dir=data_cfg.data_root / 'predictor_t', 
    device=device)
decoder_val_loader = TrainingLoader(
    batch_size=decoder_cfg.batch_size,
    split='val', data_dir=data_cfg.data_root / 'predictor_t',
    device=device)

decoder, decoder_results = train_linear_decoder(model, decoder_train_loader, decoder_val_loader, seq_banks, target_bank, model_cfg, device, data_cfg.checkpoint_dir, decoder_cfg, use_amp=USE_AMP, use_fused_optimizer=USE_FUSED)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(decoder_results['loss_history'])
plt.title('Decoder Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

## AC Evals

In [ ]:
eval_config = {
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': ac_cfg.batch_size, 'seed': SEED
}
model.eval()
eval_ctx = EvalContext(config=eval_config, data_root=data_cfg.data_root, checkpoint_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir)
eval_ctx._biojepa = model
eval_ctx._decoder = decoder
full_eval_results = run_ac_evals(eval_ctx)
eval_ctx._biojepa = None

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

In [ ]:
del decoder_train_loader, decoder_val_loader, eval_ctx
gc.collect()
torch.cuda.empty_cache()

## Summary

In [ ]:
print('=== Training Complete ===')
print(f'Encoder training final loss: {pt_results["final_loss"]:.5f}')
print(f'Composer training final loss: {align_results["final_loss"]:.5f}')
print(f'AC training final loss: {full_results["final_loss"]:.5f}')
print(f'Decoder final loss: {decoder_results["final_loss"]:.5f}')
print(f'\nCheckpoints saved to: {data_cfg.checkpoint_dir}')
print(f'Eval reports saved to: {data_cfg.eval_results_dir}')